# Single-Cell Batch Correction and Dataset Integration

**Tier 3 — Applied Bioinformatics | Module 31 · Notebook 3**

*Prerequisites: Module 30 (scRNA-seq), Notebook 2 (CITE-seq)*

---

**By the end of this notebook you will be able to:**
1. Diagnose batch effects in scRNA-seq data using mixing metrics and kBET
2. Apply Harmony, scVI, and Seurat CCA integration to correct batch effects
3. Evaluate integration quality: cell type mixing vs biological signal preservation
4. Perform label transfer between reference atlas and query datasets
5. Map query cells onto a reference atlas (Azimuth / CELLxGENE Census)


**Key resources:**
- [scib benchmarking (Luecken et al. 2022)](https://www.nature.com/articles/s41592-021-01336-8)
- [Harmony documentation](https://portals.broadinstitute.org/harmony/)
- [scVI-tools documentation](https://docs.scvi-tools.org/)
- [Azimuth reference mapping](https://azimuth.hubmapconsortium.org/)

---[< Previous: CITE-seq and Multiome Data Integration](02_cite_seq_integration.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: WGBS/RRBS Processing with Bismark >](../32_DNA_Methylation_Analysis/01_wgbs_bismark.ipynb)---

## 1. Diagnosing Batch Effects

### What is a batch effect?
A batch effect is any systematic technical variation that is correlated with a non-biological variable: different days of library preparation, different operators, different 10x kits, different sequencing runs, or different laboratories. Batch effects manifest as cells from different batches separating in UMAP space even when they should be the same cell type.

### Why batch effects are severe in scRNA-seq
- Minute differences in cell viability at capture time affect thousands of stress-response genes simultaneously
- Different ambient RNA concentrations between batches contaminate different cell fractions
- Protocol differences (enzyme lots, temperature) affect cDNA amplification non-uniformly
- Even samples processed on the same day but on different chips can show separation

### Visual diagnosis
The first diagnostic: color your UMAP by batch. If clusters separate by batch (a distinct "batch shape" overlapping true biology), you have a batch effect requiring correction.

**Red flags:**
- Cluster that contains cells from only one batch
- Same cell type in two batches forms two separate UMAP blobs
- Leiden clustering gives clusters that are 90%+ from one batch

### Quantitative metrics
Simply looking at UMAP is subjective. Quantitative metrics give objective measurements:

**LISI (Local Inverse Simpson's Index, Korsunsky et al. 2019)**:
- For each cell, look at its k nearest neighbors
- Compute the inverse Simpson's diversity index of batch labels among those neighbors
- High LISI (close to number of batches) = well-mixed = good integration
- Low LISI (close to 1) = neighbors are all from same batch = segregated batches

**ASW (Average Silhouette Width)**:
- For batch integration: lower ASW-batch = more mixed (desired)
- For cell type preservation: higher ASW-celltype = cleaner separation (desired)
- These two objectives conflict — good integration must balance both

**kBET (k-nearest neighbor Batch Effect Test)**:
- For each cell, test whether batch composition in its k-NN matches the global batch composition (chi-square test)
- Reports rejection rate: high = significant batch effect; low = well-integrated

## 2. Harmony Integration

### How Harmony works (Korsunsky et al. 2019, *Nature Methods*)
Harmony operates in PCA space and iteratively adjusts the PCA embedding to remove batch effects while preserving biological structure:

1. **Initialize**: start with the standard PCA embedding
2. **Cluster** cells into soft clusters (fuzzy k-means) based on current embedding
3. **Compute correction**: for each cluster, compute the batch centroid offsets — how much cells from Batch B are shifted relative to Batch A in that cluster
4. **Apply correction**: shift cells to remove the batch offset, merging cells from different batches that belong to the same cluster
5. **Iterate** until convergence (typically 10 iterations)

The result is a corrected PCA embedding (`adata.obsm['X_pca_harmony']`) that retains cell type structure while removing batch separation.

### Key properties of Harmony
- **In-memory**: operates on the PCA matrix, not the full count matrix → very fast (seconds to minutes)
- **Preserves sparse cell types**: cells from rare populations in one batch won't be overcorrected because the soft clusters ensure they influence only their own cluster's correction
- **No data imputation**: Harmony does NOT modify the count matrix — it only adjusts the embedding. Differential expression must still be done with the raw counts.
- **Python package**: `harmonypy` (install: `pip install harmonypy`)

### When Harmony may fail
- **Compositional imbalance**: if Batch A has only T cells and Batch B has only B cells, Harmony cannot find corresponding cell types to align — the two batches simply cannot be integrated
- **Very strong batch effects**: technical variation larger than biological variation may require deeper models (scVI)
- **Dataset-specific effects**: when batches have different cell type frequencies AND strong batch effects, over-correction may merge truly different populations

### Harmony pitfall: over-correction
Setting `theta` too high (the diversity penalty) can force batch mixing even when batches genuinely contain different cell types (e.g., disease vs healthy with truly different immune compositions). Recommended `theta` range: 1–3 (default = 2).

In [ ]:
# Harmony batch correction: implemented from scratch to show the core idea
# (Production: use harmonypy — pip install harmonypy)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

def harmony_correct(X_pca, batch_labels, n_clusters=20, max_iter=10, theta=2.0, sigma=0.1):
    """
    Simplified Harmony: iterative soft-clustering batch correction in PCA space.
    X_pca: (n_cells, n_pcs) — initial PCA embedding
    batch_labels: array of batch identifiers per cell
    """
    X = X_pca.copy()
    unique_batches = np.unique(batch_labels)
    n_batches = len(unique_batches)
    batch_matrix = np.column_stack([(batch_labels == b).astype(float) for b in unique_batches])
    
    for iteration in range(max_iter):
        # --- Step 1: Soft cluster assignment (fuzzy k-means) ---
        kmeans = KMeans(n_clusters=n_clusters, random_state=iteration, n_init=1)
        hard_assign = kmeans.fit_predict(X)
        centroids = kmeans.cluster_centers_
        
        # Soft assignment via inverse distance
        dists = np.array([np.linalg.norm(X - c, axis=1) for c in centroids]).T  # (n_cells, K)
        soft_assign = np.exp(-dists**2 / (2 * sigma**2))
        soft_assign /= soft_assign.sum(axis=1, keepdims=True) + 1e-10
        
        # --- Step 2: Compute batch centroids per cluster ---
        # For each cluster k and batch b: mean position of cells from batch b in cluster k
        correction = np.zeros_like(X)
        for k in range(n_clusters):
            weights = soft_assign[:, k]  # per-cell weight for cluster k
            cluster_center = (X * weights[:, None]).sum(0) / (weights.sum() + 1e-10)
            
            for b_idx, b in enumerate(unique_batches):
                b_mask = batch_labels == b
                b_weights = weights * b_mask
                if b_weights.sum() < 1e-6:
                    continue
                b_center = (X * b_weights[:, None]).sum(0) / (b_weights.sum() + 1e-10)
                # Correction: move cells toward cluster center, away from batch-specific center
                offset = cluster_center - b_center
                correction[b_mask] += (weights[b_mask, None] * offset[None, :] * theta / n_batches)
        
        X = X + correction * 0.3  # learning rate
    
    return X

# Apply simplified Harmony
print("Running simplified Harmony correction...")
X_pca_harmony = harmony_correct(
    adata.obsm['X_pca'][:, :20], 
    batches_arr, 
    n_clusters=20, max_iter=8, theta=2.0
)
adata.obsm['X_pca_harmony'] = X_pca_harmony
print("Done.")

# UMAP after correction
try:
    from umap import UMAP
    X_umap_post = UMAP(n_neighbors=15, min_dist=0.3, random_state=42).fit_transform(X_pca_harmony)
    adata.obsm['X_umap_harmony'] = X_umap_post

    fig, axes = plt.subplots(2, 2, figsize=(14, 11))
    batch_colors = {'Batch_A':'#e41a1c','Batch_B':'#377eb8','Batch_C':'#4daf4a'}
    ct_colors = {'T_cell':'#e41a1c','B_cell':'#377eb8','Monocyte':'#4daf4a','NK_cell':'#984ea3'}
    
    X_umap_pre = adata.obsm['X_umap_uncorrected']
    
    for batch in batch_effects:
        mask = batches_arr == batch
        axes[0,0].scatter(X_umap_pre[mask,0], X_umap_pre[mask,1],
                          c=batch_colors[batch], label=batch, s=6, alpha=0.6)
        axes[1,0].scatter(X_umap_post[mask,0], X_umap_post[mask,1],
                          c=batch_colors[batch], label=batch, s=6, alpha=0.6)
    
    for ct in cell_types:
        mask = labels_arr == ct
        axes[0,1].scatter(X_umap_pre[mask,0], X_umap_pre[mask,1],
                          c=ct_colors[ct], label=ct, s=6, alpha=0.6)
        axes[1,1].scatter(X_umap_post[mask,0], X_umap_post[mask,1],
                          c=ct_colors[ct], label=ct, s=6, alpha=0.6)
    
    axes[0,0].set_title('BEFORE correction: by batch'); axes[0,0].legend(fontsize=8)
    axes[0,1].set_title('BEFORE correction: by cell type'); axes[0,1].legend(fontsize=8)
    axes[1,0].set_title('AFTER Harmony: by batch (should be mixed)'); axes[1,0].legend(fontsize=8)
    axes[1,1].set_title('AFTER Harmony: by cell type (should be preserved)'); axes[1,1].legend(fontsize=8)
    for ax in axes.flatten():
        ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
    plt.tight_layout()
    plt.savefig('harmony_correction.png', dpi=100, bbox_inches='tight')
    plt.show()
except ImportError:
    print("umap-learn not installed")

# Quantify improvement
lisi_batch_post = compute_lisi(X_pca_harmony, batches_arr, n_neighbors=30)
lisi_ct_post    = compute_lisi(X_pca_harmony, labels_arr, n_neighbors=30)
print(f"\nBefore Harmony: LISI-batch={lisi_batch.mean():.3f}, LISI-celltype={lisi_ct.mean():.3f}")
print(f"After  Harmony: LISI-batch={lisi_batch_post.mean():.3f}, LISI-celltype={lisi_ct_post.mean():.3f}")
print(f"  -> LISI-batch increase = better batch mixing")
print(f"  -> LISI-celltype should remain similar = cell types preserved")

## 3. scVI: Deep Generative Integration

### What is scVI? (Lopez et al. 2018, *Nature Methods*)
scVI (Single-Cell Variational Inference) is a **variational autoencoder (VAE)** trained on raw count data. Unlike Harmony (which works in PCA space), scVI learns a probabilistic model directly from counts, using a negative binomial likelihood to account for overdispersion.

### Architecture
- **Encoder**: 2-layer MLP (gene expression → 128-dim latent) with batch and library size as conditional inputs
- **Latent space**: 10-dimensional Gaussian distribution (mean + variance)
- **Decoder**: reconstructs expected counts per gene, with batch-specific dispersion

The batch label is fed as a covariate to the encoder and decoder. During training, the model is penalized when the latent space retains batch information — it must encode biology, not batch.

### scVI advantages over Harmony
- **Probabilistic**: gives uncertainty estimates for each cell's position
- **Handles raw counts**: no need to normalize/log-transform before scVI
- **Native differential expression**: scVI can compute DE on the latent space using normalized expression estimates, properly accounting for batch
- **Scalable**: supports GPU acceleration; handles millions of cells with data loaders

### scVI installation and usage
```bash
pip install scvi-tools
```

```python
import scvi

# Prepare data: scVI works on raw integer counts
scvi.model.SCVI.setup_anndata(adata, batch_key='batch', layer='raw_counts')

# Build and train the model
model = scvi.model.SCVI(adata, n_layers=2, n_latent=10, gene_likelihood='nb')
model.train(max_epochs=400, early_stopping=True)

# Get batch-corrected latent representation
adata.obsm['X_scVI'] = model.get_latent_representation()

# Differential expression (handles batch properly)
de_df = model.differential_expression(
    adata,
    groupby='cell_type',
    group1='T_cell', group2='B_cell',
    delta=0.25  # minimum effect size (log2FC)
)
```

### scANVI: labeled integration
scANVI (scVI + semi-supervision) extends scVI by using available cell type labels as training signal. Labeled cells help anchor the latent space, making integration more biologically meaningful. Used for label transfer between datasets.

### BBKNN: graph-based integration
BBKNN (Batch Balanced k-NN, Polański et al. 2020) works differently:
1. For each cell, find k nearest neighbors from EACH batch separately
2. Merge these batch-balanced neighborhoods into a single graph
3. Run UMAP on this graph

This ensures each cell has neighbors from all batches, forcing cross-batch connections. Very fast (faster than Harmony), works directly in PCA space.

## 4. Label Transfer

### What is label transfer?
Label transfer is a supervised method for annotating cells in an **unlabeled query** dataset using a **labeled reference** dataset. Instead of re-annotating each new dataset from scratch, you transfer the cell type labels from a well-curated reference.

### Methods for label transfer

**KNN-based transfer (simplest)**:
For each query cell, find its k nearest neighbors in the reference dataset's embedding. The plurality label among neighbors is transferred. Confidence = fraction of neighbors with that label.

**Seurat CCA (anchor-based)**:
Seurat v3/v4 finds "anchor" cells — pairs of cells across datasets that are mutual nearest neighbors in the integrated embedding. These anchors serve as bridges for label transfer, weighting nearby cells more strongly.

**scANVI (probabilistic)**:
scANVI trains a semi-supervised VAE using reference labels. After training, it predicts labels for query cells with posterior probability estimates.

### Label transfer workflow
```python
# Using scikit-learn's KNN for simple label transfer
from sklearn.neighbors import KNeighborsClassifier

# Train on reference (labeled)
knn_classifier = KNeighborsClassifier(n_neighbors=15, weights='distance')
knn_classifier.fit(X_ref_embedding, ref_labels)

# Predict on query (unlabeled)
query_labels_predicted = knn_classifier.predict(X_query_embedding)
query_confidence = knn_classifier.predict_proba(X_query_embedding).max(axis=1)

# Filter by confidence
high_conf = query_confidence > 0.7
print(f"High-confidence predictions: {high_conf.mean():.1%}")
```

### Interpreting prediction scores
- **Score > 0.7**: reliable assignment
- **Score 0.4–0.7**: ambiguous; inspect marker genes manually
- **Score < 0.4**: cell may be a novel type not in the reference, a transitional state, or poor quality

### Azimuth reference mapping
Azimuth (Hao et al.) provides pre-trained Seurat reference atlases:
- `pbmc_multimodal`: 160k PBMCs with 30 fine-grained cell subtypes
- `lung_ref`: lung cell atlas with 58 cell types
- `bonemarrow_ref`: hematopoietic hierarchy

Azimuth is also available as a web tool (azimuth.hubmapconsortium.org) for small datasets without coding.

## 5. Integration Benchmarking and Method Selection

### The scib benchmarking framework (Luecken et al. 2022, *Nature Methods*)
scib systematically tested 16 integration methods on 77 tasks. Key findings:
1. **No single method is best**: performance varies by dataset size, batch heterogeneity, and cell type composition
2. **scVI** achieves best overall performance when GPU resources are available
3. **Harmony** is the best-performing fast method (runs in minutes, no GPU)
4. **BBKNN** is the fastest but slightly underperforms Harmony
5. **Seurat CCA** works well for cross-dataset integration but is slower

### Two conflicting objectives in integration benchmarking
Good integration must simultaneously achieve:
- **Batch removal**: batch labels should not be predictable from the embedding
- **Biological conservation**: cell type structure must be preserved

These are inherently in tension — aggressive batch correction risks merging distinct cell types.

### Metrics summary
| Metric | Measures | Range | Desired |
|--------|---------|-------|---------|
| LISI (batch) | Batch mixing | 1 to n_batches | High |
| LISI (cell type) | Cell type separation | 1 to n_types | High |
| ASW (batch) | Batch silhouette | -1 to 1 | Low |
| ASW (cell type) | Cell type silhouette | -1 to 1 | High |
| kBET | Local batch composition | 0 to 1 | Low rejection rate |

### When to skip batch correction
Not all datasets with multiple batches need correction:
- If the UMAP shows clear intermixing of batches within clusters, the data is already well-integrated
- If the experiment is designed to FIND differences between conditions, over-correcting removes the biology you want to see
- **Rule**: only correct for batch effects, not biological effects. Use samples from the same tissue type and healthy donors to establish batch-only variation before deciding on a correction strategy.

### Choosing a method for your dataset
| Scenario | Recommended | Reason |
|----------|------------|--------|
| Quick exploration | Harmony | Fast, good performance, runs in memory |
| Large dataset (>100k cells) | Harmony or scVI | scVI needs GPU, BBKNN RAM-intensive |
| Cross-species integration | scVI | Handles non-overlapping feature spaces |
| Strong batch + low cell type diversity | scVI | Better model of count distributions |
| Label transfer to reference atlas | scANVI | Probabilistic with uncertainty |

In [ ]:
# Integration benchmarking: compute multiple metrics and visualize comparison
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import silhouette_score, silhouette_samples

def compute_asw(X_emb, labels, metric='euclidean'):
    """Average Silhouette Width for a set of labels."""
    try:
        from sklearn.metrics import silhouette_score
        return silhouette_score(X_emb, labels, metric=metric)
    except Exception:
        return 0.0

# Compute metrics for each integration method
methods = {
    'Uncorrected':  adata.obsm['X_pca'][:, :20],
    'Harmony':      adata.obsm['X_pca_harmony'],
}

results = []
for method_name, X_emb in methods.items():
    lisi_b = compute_lisi(X_emb, batches_arr, n_neighbors=30)
    lisi_c = compute_lisi(X_emb, labels_arr,  n_neighbors=30)
    asw_batch = compute_asw(X_emb, batches_arr)
    asw_ct    = compute_asw(X_emb, labels_arr)
    
    results.append({
        'method': method_name,
        'LISI_batch (↑)':     lisi_b.mean(),
        'LISI_celltype (↑)':  lisi_c.mean(),
        'ASW_batch (↓)':      asw_batch,
        'ASW_celltype (↑)':   asw_ct,
    })

results_df = pd.DataFrame(results).set_index('method')
print("Integration benchmark results:")
print(results_df.round(3))

# Radar / bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Normalize metrics to [0, 1] scale for comparison
# LISI: normalize by max possible value
lisi_b_norm = results_df['LISI_batch (↑)'] / len(batch_effects)
lisi_c_norm = results_df['LISI_celltype (↑)'] / len(cell_types)
# ASW: map to 0-1 where 1=best
asw_b_norm = 1 - (results_df['ASW_batch (↓)'] + 1) / 2  # lower ASW-batch = better
asw_c_norm = (results_df['ASW_celltype (↑)'] + 1) / 2    # higher ASW-celltype = better

# Overall score: average of all normalized metrics
overall = (lisi_b_norm + lisi_c_norm + asw_b_norm + asw_c_norm) / 4

x = np.arange(len(results_df))
width = 0.35
methods_list = results_df.index.tolist()
colors_met = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
metric_labels = ['LISI batch\n(mixing)', 'LISI celltype\n(separation)',
                 'ASW batch inv.\n(low batch sep.)', 'ASW celltype\n(high bio sep.)']
metric_data = [lisi_b_norm, lisi_c_norm, asw_b_norm, asw_c_norm]

for i, (label, data) in enumerate(zip(metric_labels, metric_data)):
    axes[0].bar(x + i * 0.18, data.values, 0.18, label=label, color=colors_met[i], alpha=0.8)
axes[0].set_xticks(x + 0.27)
axes[0].set_xticklabels(methods_list, rotation=0)
axes[0].set_ylabel('Normalized score (0-1, higher=better)')
axes[0].set_title('Per-metric benchmark scores')
axes[0].legend(fontsize=8, bbox_to_anchor=(1.02, 1))
axes[0].set_ylim(0, 1.1)

axes[1].bar(methods_list, overall.values, color=['#cccccc', '#377eb8'][:len(methods_list)], edgecolor='black', alpha=0.8)
axes[1].set_title('Overall integration score\n(average of normalized metrics)')
axes[1].set_ylabel('Overall score (0-1)')
axes[1].set_ylim(0, 1)
for i, v in enumerate(overall.values):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('integration_benchmark.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nConclusion: Harmony improves batch mixing (LISI-batch) while")
print("maintaining or improving cell type separation (LISI-celltype)")
print("\nFor datasets with GPU resources, scVI would typically score higher.")
print("For quick analyses on standard hardware, Harmony is the recommended choice.")

## 3. scVI Deep Generative Integration

> VAE model accounting for batch as a covariate. scVI latent space for visualization and differential expression. Handles count data natively (no log normalization needed).

## 4. Label Transfer with Seurat or scANVI

> Transfer cell type annotations from labeled reference to unlabeled query. Prediction score thresholding. Validation against known markers.

## 5. Azimuth Reference Mapping

> Human Cell Atlas references (PBMC, lung, kidney). Map query cells. Interpret mapping score and uncertainty. Differential abundance testing (DAseq, Milo).

---[< Previous: CITE-seq and Multiome Data Integration](02_cite_seq_integration.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: WGBS/RRBS Processing with Bismark >](../32_DNA_Methylation_Analysis/01_wgbs_bismark.ipynb)---

## Summary

**Integration methods compared:**

| Method | Speed | Requires | Best scenario |
|--------|-------|---------|---------------|
| Harmony | Fast (seconds) | RAM only | Standard integration, PBMC, atlas |
| BBKNN | Fast | RAM only | Exploratory analysis |
| scVI | Moderate (GPU) | GPU + raw counts | Complex batch, probabilistic DE |
| scANVI | Moderate (GPU) | GPU + some labels | Label transfer, semi-supervised |
| Seurat CCA | Slow | R | Cross-platform RNA-seq |

**Key decisions checklist:**
1. **Visualize first**: UMAP colored by batch — do batches separate within cell type clusters?
2. **Quantify**: LISI-batch before correction to confirm problem is real
3. **Choose method**: start with Harmony for speed; upgrade to scVI if Harmony fails
4. **Validate**: LISI-batch should improve; LISI-celltype must NOT decrease significantly
5. **Never correct for condition effects**: disease vs healthy differences may be real — do not integrate across conditions unless you have verified same cell type proportions

**After integration:**
- Run Leiden clustering on corrected embedding (Harmony PCA or scVI latent)
- Compute differential expression on RAW counts (not on corrected embedding)
- Save corrected embedding: `adata.obsm['X_pca_harmony']` is your new PCA for all downstream steps